In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer
import joblib
import os

In [2]:
df = pd.read_csv("../data/parkinsons_tabular/parkinsons.csv")
print(df.shape)
print(df.columns.tolist())
print(df["status"].value_counts())

(195, 24)
['name', 'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'status', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']
status
1    147
0     48
Name: count, dtype: int64


In [3]:
# Select only the features your extract_features() also produces
df_model = df[[
    "MDVP:Fo(Hz)",       # → pitch
    "HNR",               # → hnr
    "MDVP:Jitter(Abs)",  # → jitter
    "MDVP:Shimmer",      # → shimmer
    "NHR",               # → bonus feature
    "RPDE",              # → bonus feature
    "DFA",               # → bonus feature
    "status"
]].copy()

# Rename to clean names
df_model.rename(columns={
    "MDVP:Fo(Hz)": "pitch",
    "HNR": "hnr",
    "MDVP:Jitter(Abs)": "jitter",
    "MDVP:Shimmer": "shimmer",
    "NHR": "nhr",
    "RPDE": "rpde",
    "DFA": "dfa",
    "status": "label"
}, inplace=True)

print(df_model.head())
print(df_model["label"].value_counts())

     pitch     hnr   jitter  shimmer      nhr      rpde       dfa  label
0  119.992  21.033  0.00007  0.04374  0.02211  0.414783  0.815285      1
1  122.400  19.085  0.00008  0.06134  0.01929  0.458359  0.819521      1
2  116.682  20.651  0.00009  0.05233  0.01309  0.429895  0.825288      1
3  116.676  20.644  0.00009  0.05492  0.01353  0.434969  0.819235      1
4  116.014  19.649  0.00011  0.06425  0.01767  0.417356  0.823484      1
label
1    147
0     48
Name: count, dtype: int64


In [4]:
feature_cols = ["pitch", "hnr", "jitter", "shimmer", "nhr", "rpde", "dfa"]

X = df_model[feature_cols]
y = df_model["label"]

imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"AUROC    : {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Parkinson's"]))

Accuracy : 0.9231
AUROC    : 0.9552
              precision    recall  f1-score   support

     Healthy       0.89      0.80      0.84        10
 Parkinson's       0.93      0.97      0.95        29

    accuracy                           0.92        39
   macro avg       0.91      0.88      0.90        39
weighted avg       0.92      0.92      0.92        39



In [5]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/parkinsons_model.pkl")
joblib.dump(imputer, "../models/parkinsons_imputer.pkl")
joblib.dump(feature_cols, "../models/parkinsons_feature_cols.pkl")
print("✅ Saved: parkinsons_model.pkl")
print("✅ Saved: parkinsons_imputer.pkl")
print("✅ Saved: parkinsons_feature_cols.pkl")

✅ Saved: parkinsons_model.pkl
✅ Saved: parkinsons_imputer.pkl
✅ Saved: parkinsons_feature_cols.pkl


In [6]:
import sys
sys.path.append("../src")
from feature_extraction.audio_features import extract_features

def predict_parkinsons(audio_path):
    # Extract features from wav
    features = extract_features(audio_path)
    
    # Use only the features this model was trained on
    # Note: rpde and dfa are NOT in your extractor, so we fill with dataset mean
    rpde_mean = df_model["rpde"].mean()
    dfa_mean  = df_model["dfa"].mean()
    
    X = pd.DataFrame([{
        "pitch":   features["pitch"],
        "hnr":     features["hnr"],
        "jitter":  features["jitter"],
        "shimmer": features["shimmer"],
        "nhr":     1 - (features["hnr"] / (features["hnr"] + 1)),  # approximate
        "rpde":    rpde_mean,
        "dfa":     dfa_mean
    }])
    
    X_imputed = imputer.transform(X)
    prediction = model.predict(X_imputed)[0]
    probability = model.predict_proba(X_imputed)[0][1]
    
    label = "⚠️ Parkinson's Risk Detected" if prediction == 1 else "✅ Healthy"
    print(f"Result     : {label}")
    print(f"Risk Score : {probability:.2%}")
    return prediction, probability

print("Function ready — pass any .wav file to predict_parkinsons(path)")

Function ready — pass any .wav file to predict_parkinsons(path)


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = cross_val_score(model, X_imputed, y, cv=skf, scoring="accuracy")
cv_auc = cross_val_score(model, X_imputed, y, cv=skf, scoring="roc_auc")

print("Parkinson's Model - 5-Fold Cross Validation:")
print(f"  Accuracy : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  AUROC    : {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

Parkinson's Model - 5-Fold Cross Validation:
  Accuracy : 0.8872 +/- 0.0384
  AUROC    : 0.9494 +/- 0.0197
